# Computational Theory Assignment Problem (SHA-256)


## Problem 1: Representing SHA-256 Data
Before we go ahead with the implementation of SHA-256, I'll first clarify a few things, mainly how input, output, and intermediate data in Python.

NOTE: the small python example I give for each data representation is just a simplified example for demo's sake, the actual implementation is after that, so keep that in mind.

### 32-bit words
Unlike C/C++/Rust where there's explicit `uint32_t` typing that I would love to use, python is dynamically typed and the way its integer behaves is not the same, it is not fixed-width, which in sha-256 implementation's case, is exactly the type of behaviour I would need to preserve. Initially, I wanted to go with a `Int32` class wrapper to make it safer, but decided against it because of a few reasons.
1. Performance overhead - sha256 produces a 32-bit result in every single operation, with a class, in the thousands of loops that gets ran in the function there would be a LOT of object allocation overhead
2. Lose out on clean bitwise op syntax - this one is especially painful because python doesn't even enforce static type bounds on a custom class, meaning the `Int32` class is just wrapping a magically expanding python int under the hood anyway, so every single math op still requires me to manually slice it back down to 32 bits. I would need to overload so many operators (i.e. `__add__`,`__xor__`, `__rshift__`, etc. you get the idea), which adds a lot of work for no reason, not to mention more places where things could go wrong. 
3. KISS principle, this one is self explanatory

So, in the end, I'll just go ahead with the native python int with a 32bit mask (`& 0xFFFFFFFF`), nice and simple, if it overflows, the overflowed bits would get masked off anyway and to make things smoother and easier to read, we can add a bunch of helper functions to keep things nice and clean.

In [ ]:
# placeholder for now

### sequences of 32-bit words
I'll be using `list[int]` to represent the sequence of words, since we're already using the native ints anyways, a typical python list should be a no brainer, it is random access, O(1) time, easy to initialize. It makes it clearer if I explain why we're NOT using the alternatives.
1. python array module - it does have a native uint 32 type, but that adds unnecessary type conversion when I put the words in or access them, once again, the KISS principle
2. tuple - tuples are immutable, we need to mutate them during a step in sha256, straightforward
3. bytes/bytearray - This is an alternative for both the 32bit word and sequence of the words, and I suppose it would work fine BUT I would need to call `int.from_bytes()` every time I want to perform addition or bitwise op on a word, this is really unnecessary.

In [ ]:
# placeholder for now

### input messages
For the input messages it's also very straightforward, just use python's immutable `bytes` type because sha256 operates at a byte level, the `bytes` type is evidently byte-level precise, I'm not going to be using characters because UTF-8 strings can range from 1-4 bytes long e.g. ඞ (0xE0 0xB6 0x9E) is 3 bytes long meanwhile the char `a` is 1 byte long. I suppose we could theoretically use `str` or `list` here too but that would require extra work. 

e.g. using `str` forces me to encode to bytes before calculating exact bit lengths or applying masks anyway

e.g.2 using `list[int]` of raw byte values wastes memory, loses the super easy byte slicing provided by `bytes` and also require extra conversion steps during the block processing, which is really unnecessary

In [ ]:
message = "abc"
message_bytes = message.encode("utf-8")

print(message_bytes)  # b'abc'
print(len(message_bytes))  # 3

### 512-bit message blocks

### the final 256-bit hash value

### big-endian integers